In [0]:
#
%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/audit_logger.ipynb

In [0]:
#
%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/audit_logger.ipynb

In [0]:
# XML PIPELINE RUNNER

# LOAD CONFIG
%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/notebooks/00_framework_config.ipynb

# LOAD UTILITIES

%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/xml_parser.ipynb

%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/schema_resolver.ipynb

%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/xml_flattener.ipynb

%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/dataframe_converter.ipynb

%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/type_casting_engine.ipynb

%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/audit_logger.ipynb

%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/file_already_processed.ipynb

%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/delta_loader.ipynb

%run /Workspace/Users/tejswi.tech2db@gmail.com/XML_Framework/utils/file_archiver.ipynb


# DISCOVER XML FILES

xml_files = [

    file.path

    for file in dbutils.fs.ls(
        XML_VOLUME
    )

    if file.path.lower().endswith(".xml")

]

print(
    f"Files Found: {len(xml_files)}"
)

for file in xml_files:

    print(file)


# ============================================
# PROCESS EACH XML FILE
# ============================================

for xml_path in xml_files:

    print(
        f"\nProcessing File: {xml_path}"
    )

    try:

        # --------------------------------------------
        # PARSE XML
        # --------------------------------------------

        xml_info = parse_xml(
            xml_path
        )

        # --------------------------------------------
        # DUPLICATE CHECK
        # --------------------------------------------

        if is_file_processed(
            xml_info["file_name"],
            f"{CATALOG}.{BRONZE_SCHEMA}.ingestion_audit"
        ):

            print(
                f"Already Processed: "
                f"{xml_info['file_name']}"
            )

            archive_file(
                xml_path,
                ARCHIVE_VOLUME
            )

            print(
                f"Archived Duplicate File: "
                f"{xml_info['file_name']}"
            )

            continue

        print(
            f"Root Element: "
            f"{xml_info['root_element']}"
        )

        # --------------------------------------------
        # RESOLVE SCHEMA
        # --------------------------------------------

        schema_df = resolve_schema(
            xml_info["root_element"],
            SCHEMA_REGISTRY_TABLE
        )

        target_table = schema_df.select(
            "target_table"
        ).first()[0]

        print(
            f"Target Table: {target_table}"
        )

        # --------------------------------------------
        # FLATTEN XML
        # --------------------------------------------

        records = flatten_orders(
            xml_info["root"]
        )

        print(
            f"Records Found: "
            f"{len(records)}"
        )

        # --------------------------------------------
        # DATAFRAME CONVERSION
        # --------------------------------------------

        df = records_to_dataframe(
            records
        )

        # --------------------------------------------
        # TYPE CASTING
        # --------------------------------------------

        typed_df = apply_type_casting(
            df,
            f"{CATALOG}.{BRONZE_SCHEMA}.{target_table}"
        )

        # --------------------------------------------
        # LOAD TO DELTA
        # --------------------------------------------

        load_to_delta(
            df=typed_df,
            target_table=
                f"{CATALOG}.{BRONZE_SCHEMA}.{target_table}"
        )

        # --------------------------------------------
        # AUDIT SUCCESS
        # --------------------------------------------

        log_ingestion(
            file_name=xml_info["file_name"],
            target_table=target_table,
            record_count=int(len(records)),
            status="SUCCESS",
            audit_table=
                f"{CATALOG}.{BRONZE_SCHEMA}.ingestion_audit",
            error_message=None
        )

        # --------------------------------------------
        # ARCHIVE FILE
        # --------------------------------------------

        archive_file(
            xml_path,
            ARCHIVE_VOLUME
        )

        print(
            f"Successfully Loaded: "
            f"{xml_info['file_name']}"
        )

    except Exception as e:

        print(
            f"FAILED: {xml_path}"
        )

        print(
            f"ERROR: {str(e)}"
        )

        try:

            file_name = (
                xml_path.split("/")[-1]
            )

            log_ingestion(
                file_name=file_name,
                target_table="UNKNOWN",
                record_count=0,
                status="FAILED",
                audit_table=
                    f"{CATALOG}.{BRONZE_SCHEMA}.ingestion_audit",
                error_message=str(e)
            )

        except Exception as audit_error:

            print(
                f"Audit Logging Failed: "
                f"{str(audit_error)}"
            )

        continue

# VERIFY FINAL TABLE

#display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{target_table}"))

In [0]:
display(
    spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.ingestion_audit"
    )
)